# Weather Pipeline Walkthrough

This notebook demonstrates the end-to-end daily weather pipeline for the configured cities.

The pipeline is:

**Open-Meteo Archive API → Python ingestion → PostgreSQL raw → dbt staging → dbt mart → data quality tests**

For this walkthrough, the logical weather date is **2026-09-10**.

The notebook reuses the same ingestion functions used by the Airflow DAG rather than implementing a separate extraction pipeline.

In [1]:
from ingestion.weather_pipeline import extract_weather, load_weather_rows

weather_date = "2026-09-10"

print(f"Using weather date: {weather_date}")

Using weather date: 2026-09-10


In [2]:
import sys

print(sys.executable)
print(sys.path)

/home/airflow/.local/bin/python
['/opt/airflow', '/usr/local/lib/python312.zip', '/usr/local/lib/python3.12', '/usr/local/lib/python3.12/lib-dynload', '', '/home/airflow/.local/lib/python3.12/site-packages']


In [3]:
rows = extract_weather(
    "/opt/airflow/config/cities.yml",
    weather_date,
)

print(f"Extracted {len(rows)} rows")
rows

Extracted 4 rows


[{'city': 'Bengaluru',
  'latitude': 12.9716,
  'longitude': 77.5946,
  'date': '2026-09-10',
  'weather_code': 51,
  'temperature_2m_max': 30.1,
  'temperature_2m_min': 21.0,
  'precipitation_sum': 1.0,
  'wind_speed_10m_max': 13.7},
 {'city': 'Mumbai',
  'latitude': 19.076,
  'longitude': 72.8777,
  'date': '2026-09-10',
  'weather_code': 51,
  'temperature_2m_max': 29.1,
  'temperature_2m_min': 25.2,
  'precipitation_sum': 2.8,
  'wind_speed_10m_max': 10.2},
 {'city': 'Delhi',
  'latitude': 28.6139,
  'longitude': 77.209,
  'date': '2026-09-10',
  'weather_code': 2,
  'temperature_2m_max': 34.4,
  'temperature_2m_min': 26.6,
  'precipitation_sum': 0.0,
  'wind_speed_10m_max': 11.5},
 {'city': 'Hyderabad',
  'latitude': 17.384,
  'longitude': 78.4564,
  'date': '2026-09-10',
  'weather_code': 61,
  'temperature_2m_max': 31.5,
  'temperature_2m_min': 25.2,
  'precipitation_sum': 3.2,
  'wind_speed_10m_max': 15.3}]

In [4]:
load_weather_rows(rows)

print("Rows loaded into PostgreSQL raw.weather_daily")

Rows loaded into PostgreSQL raw.weather_daily


In [5]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="warehouse",
    user="de",
    password="de",
)

cursor = conn.cursor()

cursor.execute(
    """
    SELECT
        city,
        date,
        temperature_2m_max,
        temperature_2m_min,
        precipitation_sum,
        wind_speed_10m_max
    FROM raw.weather_daily
    WHERE date = %s
    ORDER BY city
    """,
    (weather_date,),
)

columns = [description[0] for description in cursor.description]
raw_df = pd.DataFrame(cursor.fetchall(), columns=columns)

cursor.close()
conn.close()

raw_df

,city,date,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max
0,Bengaluru,2026-09-10,30.1,21.0,1.0,13.7
1,Delhi,2026-09-10,34.4,26.6,0.0,11.5
2,Hyderabad,2026-09-10,31.5,25.2,3.2,15.3
3,Mumbai,2026-09-10,29.1,25.2,2.8,10.2


In [6]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="warehouse",
    user="de",
    password="de",
)

cursor = conn.cursor()

cursor.execute(
    """
    SELECT
        city,
        date,
        temperature_2m_max,
        temperature_2m_min,
        precipitation_sum,
        wind_speed_10m_max
    FROM staging.stg_weather
    WHERE date = %s
    ORDER BY city
    """,
    (weather_date,),
)

columns = [description[0] for description in cursor.description]
staging_df = pd.DataFrame(cursor.fetchall(), columns=columns)

cursor.close()
conn.close()

staging_df

,city,date,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max
0,Bengaluru,2026-09-10,30.1,21.0,1.0,13.7
1,Delhi,2026-09-10,34.4,26.6,0.0,11.5
2,Hyderabad,2026-09-10,31.5,25.2,3.2,15.3
3,Mumbai,2026-09-10,29.1,25.2,2.8,10.2


In [7]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="warehouse",
    user="de",
    password="de",
)

cursor = conn.cursor()

cursor.execute(
    """
    SELECT
        city,
        date,
        temperature_2m_max,
        temperature_2m_min,
        temperature_2m_avg,
        precipitation_sum
    FROM marts.fct_city_daily
    WHERE date = %s
    ORDER BY city
    """,
    (weather_date,),
)

columns = [description[0] for description in cursor.description]
mart_df = pd.DataFrame(cursor.fetchall(), columns=columns)

cursor.close()
conn.close()

mart_df

,city,date,temperature_2m_max,temperature_2m_min,temperature_2m_avg,precipitation_sum
0,Bengaluru,2026-09-10,30.1,21.0,25.55,1.0
1,Delhi,2026-09-10,34.4,26.6,30.50,0.0
2,Hyderabad,2026-09-10,31.5,25.2,28.35,3.2
3,Mumbai,2026-09-10,29.1,25.2,27.15,2.8


In [8]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    dbname="warehouse",
    user="de",
    password="de",
)

cursor = conn.cursor()

cursor.execute(
    """
    SELECT
        city,
        date,
        COUNT(*) AS row_count
    FROM raw.weather_daily
    WHERE date = %s
    GROUP BY city, date
    ORDER BY city
    """,
    (weather_date,),
)

columns = [description[0] for description in cursor.description]
rerun_df = pd.DataFrame(cursor.fetchall(), columns=columns)

cursor.close()
conn.close()

rerun_df

,city,date,row_count
0,Bengaluru,2026-09-10,1
1,Delhi,2026-09-10,1
2,Hyderabad,2026-09-10,1
3,Mumbai,2026-09-10,1


## Data quality checks

The dbt pipeline includes tests for required fields and uniqueness of the city/date grain.

The tests are run by the Airflow DAG after the dbt models are built.

In [9]:
import subprocess

result = subprocess.run(
    ["dbt", "test"],
    cwd="/opt/airflow/dbt",
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("dbt tests failed")

10:46:49  Running with dbt=1.8.8
10:46:49  Registered adapter: postgres=1.8.2
10:46:50  Found 2 models, 4 data tests, 1 source, 424 macros
10:46:50  
10:46:50  Concurrency: 2 threads (target='dev')
10:46:50  
10:46:50  1 of 4 START test fct_city_daily_unique ........................................ [RUN]
10:46:50  2 of 4 START test not_null_fct_city_daily_city ................................. [RUN]
10:46:50  2 of 4 PASS not_null_fct_city_daily_city ....................................... [PASS in 0.14s]
10:46:50  1 of 4 PASS fct_city_daily_unique .............................................. [PASS in 0.14s]
10:46:50  3 of 4 START test not_null_fct_city_daily_date ................................. [RUN]
10:46:51  4 of 4 START test not_null_fct_city_daily_temperature_2m_avg ................... [RUN]
10:46:51  4 of 4 PASS not_null_fct_city_daily_temperature_2m_avg ......................... [PASS in 0.08s]
10:46:51  3 of 4 PASS not_null_fct_city_daily_date ...............................

## Example mart result

As a simple downstream use case, we can identify the city with the highest average temperature for the selected date.

In [10]:
hottest_city = mart_df.loc[
    mart_df["temperature_2m_avg"].idxmax()
]

print(
    f"On {weather_date}, {hottest_city['city']} "
    f"had the highest average temperature: "
    f"{hottest_city['temperature_2m_avg']:.2f}°C"
)

On 2026-09-10, Delhi had the highest average temperature: 30.50°C


## Pipeline design choices

- **Raw layer:** API fields are kept unmodified in `raw.weather_daily`. The table also stores city coordinates and `loaded_at` as pipeline metadata.
- **Idempotency:** `(city, date)` is the primary key, and the load uses PostgreSQL `ON CONFLICT DO UPDATE`. Rerunning the same logical date therefore updates existing records instead of creating duplicates.
- **Transformations:** dbt owns the transformation logic. The staging model reads from the raw source, while the mart derives `temperature_2m_avg`.
- **Data quality:** dbt tests validate the city/date grain and required fields.
- **Orchestration:** Airflow uses the logical date for extraction, runs daily, supports backfills, and executes `dbt run` followed by `dbt test`.
- **Notebook:** This walkthrough reuses the same ingestion functions used by the Airflow DAG and queries the resulting database layers for verification.